# Final Comparative Evaluation
Combines baseline ML results with transformer results into a final comparison table and charts.

In [ ]:
# ==========================================
# IMPORTS
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('../results/graphs', exist_ok=True)
os.makedirs('../results/reports', exist_ok=True)

In [ ]:
# ==========================================
# LOAD BASELINE RESULTS
# Generated by your teammate's baseline notebook.
# ==========================================
baseline_df = pd.read_csv('../results/baseline_results.csv')
print('Baseline results:')
print(baseline_df)

In [ ]:
# ==========================================
# LOAD TRANSFORMER RESULTS
# Generated by transformers.ipynb
# ==========================================
transformer_df = pd.read_csv('../results/transformer_results.csv')
print('Transformer results:')
print(transformer_df)

In [ ]:
# ==========================================
# ALIGN COLUMNS AND COMBINE
# Standardise column names across both files.
# ==========================================

# Rename transformer result columns (Trainer adds eval_ prefix)
transformer_df = transformer_df.rename(columns={
    'eval_accuracy':  'Accuracy',
    'eval_f1':        'F1',
    'eval_precision': 'Precision',
    'eval_recall':    'Recall',
})

# Keep only needed columns
keep_cols = ['Model', 'Accuracy', 'F1', 'Precision', 'Recall']

# Filter to available columns only (handle edge cases)
baseline_clean   = baseline_df[[c for c in keep_cols if c in baseline_df.columns]]
transformer_clean = transformer_df[[c for c in keep_cols if c in transformer_df.columns]]

combined = pd.concat([baseline_clean, transformer_clean], ignore_index=True)

# Round all numeric columns
for col in ['Accuracy', 'F1', 'Precision', 'Recall']:
    if col in combined.columns:
        combined[col] = combined[col].round(4)

print('\n=== FULL COMPARISON TABLE ===')
print(combined.to_string(index=False))

In [ ]:
# ==========================================
# SAVE FINAL TABLE
# ==========================================
combined.to_csv('../results/reports/final_comparison.csv', index=False)
print('Final comparison saved to ../results/reports/final_comparison.csv')

In [ ]:
# ==========================================
# BAR CHART — F1 Score Comparison
# ==========================================
plt.figure(figsize=(10, 5))

colors = ['steelblue'] * len(baseline_clean) + ['seagreen'] * len(transformer_clean)

bars = plt.bar(combined['Model'], combined['F1'], color=colors, edgecolor='black', width=0.5)

# Add value labels on top
for bar, val in zip(bars, combined['F1']):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{val:.3f}',
        ha='center', va='bottom', fontsize=9
    )

plt.title('Model Comparison — Weighted F1 Score', fontsize=13)
plt.xlabel('Model')
plt.ylabel('Weighted F1 Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=20, ha='right')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Baseline ML'),
    Patch(facecolor='seagreen',  label='Transformer')
]
plt.legend(handles=legend_elements)

plt.tight_layout()
plt.savefig('../results/graphs/model_comparison_f1.png', dpi=150)
plt.show()
print('F1 comparison chart saved.')

In [ ]:
# ==========================================
# GROUPED BAR CHART — All Metrics
# ==========================================
metrics = ['Accuracy', 'F1', 'Precision', 'Recall']
metrics = [m for m in metrics if m in combined.columns]

plot_df = combined.set_index('Model')[metrics]

ax = plot_df.plot(kind='bar', figsize=(12, 5), width=0.7, edgecolor='black')
plt.title('All Models — All Metrics Comparison', fontsize=13)
plt.ylabel('Score')
plt.xlabel('Model')
plt.ylim(0, 1.1)
plt.xticks(rotation=20, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../results/graphs/model_comparison_all_metrics.png', dpi=150)
plt.show()
print('Full metrics chart saved.')

In [ ]:
# ==========================================
# PRINT FINAL SUMMARY
# ==========================================
best_model = combined.loc[combined['F1'].idxmax()]

print('=' * 50)
print('FINAL EVALUATION SUMMARY')
print('=' * 50)
print(combined.to_string(index=False))
print()
print(f'Best Model:  {best_model["Model"]}')
print(f'Best F1:     {best_model["F1"]}')
print(f'Best Acc:    {best_model["Accuracy"]}')
print('=' * 50)